<a href="https://colab.research.google.com/github/Adhiaris/Scikit-Learn-Cookbook/blob/main/1_API_Elements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1: scikit-learn Core API

This chapter introduces the fundamental mechanics of **scikit-learn**. We will explore how its unified interface operates across various algorithms.

**Key takeaways:**
- Understanding the standardized API design.
- Utilizing `fit()` and `predict()` for models.
- Applying `fit()` and `transform()` for preprocessing.
- Creating custom components and leveraging pipelines.
- Accessing attributes, metrics, and managing hyperparameters.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.6.1
NumPy version: 2.0.2


Note: The code assumes scikit-learn **1.8.0**, taking advantage of the latest metadata routing features.

## API Design Philosophy

The strength of scikit-learn lies in its **standardized architecture** based on:
1. **Consistency:** All algorithms use identical methods.
2. **Simplicity:** Sensible default parameters streamline prototyping.
3. **Modularity:** Pipelines make it easy to chain separate steps.
4. **Reusability:** Custom code easily integrates into the ecosystem.

This allows practitioners to swap algorithms with minimal code adjustments.

## Core Estimators

An **estimator** is an object capable of learning from data. Supervised estimators map features $\mathbf{X}$ to targets $\mathbf{y}$ via the `fit()` method and generate new values using `predict()`.

### Example: Linear Regression

Linear regression identifies optimal weights to minimize error between predicted and actual values.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Predictions for X = [6, 7]:", predictions)

Predictions for X = [6, 7]: [5.75 6.7 ]


The estimator successfully derived the linear relationship. This consistent `instantiate, fit, predict` workflow applies to all supervised models within the library.

### Unsupervised Learning with `fit_predict()`

For tasks where no target labels exist, the `fit_predict()` method efficiently trains the model and generates cluster assignments in one step.

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Cluster labels:", labels)
print("Cluster centers:", kmeans.cluster_centers_.flatten())

Cluster labels: [0 0 0 1 1]
Cluster centers: [2.  4.5]


K-Means effectively split the data. Remember to use `fit_predict` mainly when working with training data in unsupervised scenarios, while supervised learning demands separate operations for train and test sets.

## Preprocessing via `transform()`

Transformers manipulate data formats rather than predicting outcomes. They leverage `fit()` to understand data distributions and `transform()` to apply changes.

### Normalizing with StandardScaler

Standardization scales features so they maintain a mean of zero and standard deviation of one, which is vital for distance-sensitive algorithms.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()

scaler.fit(X)

X_scaled = scaler.transform(X)
print("Original data:")
print(X)
print()
print("Scaled data:")
print(X_scaled)
print()
print("Mean of each feature (learned):", scaler.mean_)
print("Std of each feature (learned):", scaler.scale_)

Original data:
[[1 2]
 [3 4]
 [5 6]]

Scaled data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean of each feature (learned): [3. 4.]
Std of each feature (learned): [1.63299316 1.63299316]


Crucially, avoid calling `fit()` on your test dataset. It will recalculate means and disrupt the established baseline, leading to data leakage.

### The `fit_transform()` Approach

Using `fit_transform()` is the standard practice for training data as it is concise and often computationally optimized.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled data (via fit_transform):")
print(X_scaled)

print()
print("Column means after scaling:", X_scaled.mean(axis=0))
print("Column stds after scaling:", X_scaled.std(axis=0))

Scaled data (via fit_transform):
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Column means after scaling: [0. 0.]
Column stds after scaling: [1. 1.]


This enforces the necessary boundary between train and test datasets, which is essential for pipeline integration.

## Building Custom Components

By extending classes like `BaseEstimator` and specific mixins (`TransformerMixin`, `ClassifierMixin`), developers can easily integrate custom logic directly into the standard workflow.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
import numpy as np

class ValueClipper(BaseEstimator, TransformerMixin):
    """Clips all feature values to [lower, upper] range."""

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        X = np.array(X)
        return np.clip(X, self.lower, self.upper)

X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
clipper.fit(X)
X_clipped = clipper.transform(X)

print("Original data:")
print(X)
print()
print("Clipped data (range [0.0, 1.0]):")
print(X_clipped)
print()
print("Parameters:", clipper.get_params())

Original data:
[[ 0.5 -0.3]
 [ 1.5  0.7]
 [ 0.2  2.1]]

Clipped data (range [0.0, 1.0]):
[[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]

Parameters: {'lower': 0.0, 'upper': 1.0}


This custom implementation ensures `fit()` returns itself for method chaining and properly utilizes `TransformerMixin` to automatically inherit `fit_transform()` capabilities.

## Pipeline Automation

Pipelines connect transformers and an estimator into a reproducible sequence. Calling `fit()` passes data down the chain automatically, mitigating data leakage errors.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

X, y = make_classification(n_samples=200, n_features=5, n_informative=3, n_redundant=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipe.fit(X_train, y_train)

train_score = pipe.score(X_train, y_train)
test_score = pipe.score(X_test, y_test)
print(f"Training accuracy: {train_score:.4f}")
print(f"Test accuracy:     {test_score:.4f}")
print(f"Pipeline steps:    {[name for name, _ in pipe.steps]}")

Training accuracy: 0.9267
Test accuracy:     0.8800
Pipeline steps:    ['scaler', 'classifier']


The test sequence enforces the strict separation between `fit_transform` (for training) and `transform` (for testing). Pipelines are highly beneficial for model serialization and safe deployment.

### MLOps Relevance

Pipelines form the bedrock of Continuous Integration and Deployment (CI/CD) environments, providing stable, versionable assets for model monitoring platforms like MLflow.

## Inspecting Model Attributes

Fitted parameters, such as slopes and intercepts, are designated by a trailing underscore (`coef_`, `intercept_`). The `score()` function provides rapid baseline evaluations.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

print("Coefficients (w):", model.coef_)

print("Intercept (b):   ", model.intercept_)

r2 = model.score(X, y)
print(f"R-squared:         {r2:.6f}")

y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nManual prediction for X=3: {model.coef_[0]:.2f} * 3 + {model.intercept_:.2f} = {y_hat:.2f}")
print(f"Actual y for X=3: {y[2]}")

Coefficients (w): [0.95]
Intercept (b):    0.04999999999999938
R-squared:         0.980978

Manual prediction for X=3: 0.95 * 3 + 0.05 = 2.90
Actual y for X=3: 3.0


Inspecting underlying values like $R^2$ gives insight into how closely predictions mirror actual data distributions. However, always exercise caution to ensure metrics match the model context.

## Managing Hyperparameters

Parameters configured before training are managed programmatically via `set_params()` and `get_params()`, establishing the groundwork for automated tuning via Grid or Randomized Search.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.set_params(n_estimators=100, max_depth=10, random_state=42)

params = model.get_params()
print("Full parameter dictionary:")
for key, value in sorted(params.items()):
    print(f"  {key}: {value}")

Full parameter dictionary:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


This dynamic parameter access is what makes broad, systematic model tuning operations possible natively within the library.

## Advanced Metadata Infrastructure

Estimators contain backend tags clarifying multi-output capacity or API constraints. This enables advanced routing of objects like sample weights deep into pipeline structures.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
tags = lr.__sklearn_tags__()

print("Estimator type:", type(lr).__name__)
print()
print("Key tags:")
print(f"  Supports multiclass:     True (via OvR or multinomial)")
print(f"  Requires fit:            True")
print(f"  Array API support:       {tags.array_api_support}")
print(f"  Non-deterministic:       {tags.non_deterministic}")

Estimator type: LogisticRegression

Key tags:
  Supports multiclass:     True (via OvR or multinomial)
  Requires fit:            True
  Array API support:       False
  Non-deterministic:       False


These tags maintain software stability by communicating estimator requirements seamlessly in the background.

## Final Takeaways

- **Stick to the Pattern:** Standardize around `fit`, `transform`, and `predict`.
- **Preprocess Wisely:** Handle raw data transformations cautiously.
- **Adopt Pipelines:** Construct robust operational blocks to stop leakage.
- **Cross-Validate Always:** Check model validity rigorously.

## Summary Overview
This lesson solidified fundamental scikit-learn interactions, mapping out estimators, transformers, routing functions, and standard metrics to facilitate high-quality analysis.